# HEILO Seed v1.0: pré-treino em português (Colab, GPU grátis)

A auditoria mostrou que o Seed v0.x (3 M parâmetros, lendo byte por byte, só ~500 conversas) **decorava** respostas e não entendia perguntas novas. O v1.0 muda a receita:

1. **Tokenizer BPE próprio da HEILO**, treinado aqui. Não é o do Qwen.
2. **Pré-treino** em texto geral em português (Wikipédia), para o modelo aprender a língua. São ~30 M parâmetros, com pesos aleatórios no início.
3. **Ajuste de conversa (SFT)** com o dataset HEILO aprovado, com parada antecipada pela validação.
4. **Avaliação** no `heilo_eval_v1` e na **sonda independente**. A versão só vira a ativa se melhorar, e a sonda não pode piorar.
5. Salva no GitHub. No PC, depois, rode `atualizar_heilo.bat`.

**Antes de começar**
- `Ambiente de execução → Alterar tipo → GPU (T4)`.
- No PC, rode `publicar_heilo.bat`, para o GitHub ficar com a versão atual da HEILO.
- Crie um token no GitHub (fine-grained, **Contents: Read and write** no repositório `HELIO`) e salve na 🔑 (Secrets) do Colab como `GITHUB_TOKEN`.
- A célula 2 pede acesso ao Google Drive. Os checkpoints ficam lá: se o Colab desconectar, é só rodar tudo de novo que ele **continua de onde parou**.

Depois é `Ambiente de execução → Executar tudo`. Leva de 2h30 a 3h30.

In [ ]:
# 1) Configuração
REPO = "hugolindo12/HELIO"
BRANCH = "main"
NOME_VERSAO = "v1.0"
MINUTOS_PRETREINO = 150       # tempo de GPU para o pré-treino
MAX_CARACTERES = 400_000_000  # ~95 milhões de tokens de Wikipédia PT
VOCAB = 16000
LOTE = 32
PASSOS_SFT = 2000

import torch, time
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA — ative a GPU!")

In [ ]:
# 2) Google Drive (checkpoints que sobrevivem a desconexões)
from google.colab import drive
drive.mount("/content/drive")
PASTA = "/content/drive/MyDrive/heilo_seed_v1"
import os; os.makedirs(PASTA, exist_ok=True)

In [ ]:
# 3) Código da HEILO (a pasta precisa se chamar 'heilo' por causa dos imports)
from google.colab import userdata
import subprocess, sys
TOKEN = userdata.get("GITHUB_TOKEN")
URL = f"https://{TOKEN}@github.com/{REPO}.git"
if not os.path.exists("/content/heilo"):
    subprocess.run(["git", "clone", "-b", BRANCH, URL, "/content/heilo"], check=True)
else:
    subprocess.run(["git", "-C", "/content/heilo", "pull"], check=True)
sys.path.insert(0, "/content")
!pip -q install datasets tokenizers
!git -C /content/heilo log --oneline -1

## Tokenizer próprio + corpus

In [ ]:
# 4) Tokenizer BPE da HEILO + corpus tokenizado (salvos no Drive; pula se já existir)
from pathlib import Path
from heilo.training import seed_v1, pretrain
from heilo.models.seed.tokenizer import BPETokenizer
tok_arq, corpus_dir = Path(PASTA) / "tokenizer.json", Path(PASTA) / "corpus"
if tok_arq.exists():
    tok = BPETokenizer.from_json(tok_arq.read_text(encoding="utf-8"))
else:
    t0 = time.time()
    amostra = list(seed_v1.textos_wikipedia(60_000_000))
    tok = seed_v1.treinar_tokenizer(amostra, seed_v1.textos_heilo(), vocab_size=VOCAB)
    tok_arq.write_text(tok.tok.to_str(), encoding="utf-8")
    del amostra
    print(f"tokenizer treinado em {time.time()-t0:.0f}s")
print("vocabulário:", tok.vocab_size, "| teste:", tok.tok.encode("Olá! Eu sou a HEILO, criada pelo Hugo.").tokens)
if not (corpus_dir / "meta.json").exists():
    t0 = time.time()
    meta = pretrain.preparar_corpus(seed_v1.textos_wikipedia(MAX_CARACTERES), tok, corpus_dir,
                                    max_tokens=10**12, log=lambda s: None)
    print(meta, f"{(time.time()-t0)/60:.1f} min")
else:
    print(open(corpus_dir / "meta.json").read())

## Pré-treino (retoma sozinho se o Colab cair)

In [ ]:
# 5) Mede a velocidade e ajusta os passos para caber no tempo
cfg = seed_v1.CONFIG_V1
ck = Path(PASTA) / "pretreino.pt"
if not ck.exists():
    t0 = time.time()
    pretrain.pretreinar(cfg, tok, corpus_dir, Path("/content/velocidade.pt"), passos=60,
                        lote=LOTE, avaliar_cada=10**9, salvar_cada=10**9, aquecimento=10, log=lambda s: None)
    seg_por_passo = (time.time() - t0) / 60
    PASSOS = int(MINUTOS_PRETREINO * 60 / seg_por_passo)
    Path(PASTA, "passos.txt").write_text(str(PASSOS))
    print(f"{seg_por_passo:.3f} s/passo → {PASSOS} passos (~{PASSOS*LOTE*cfg.block_size/1e6:.0f} M tokens)")
PASSOS = int(Path(PASTA, "passos.txt").read_text())

In [ ]:
# 6) Pré-treino
res_pre = pretrain.pretreinar(cfg, tok, corpus_dir, ck, passos=PASSOS, lote=LOTE,
                              tempo_max_min=MINUTOS_PRETREINO + 20, avaliar_cada=500, salvar_cada=1000)
print("pré-treino:", res_pre["passo"], "passos |", res_pre["hist"][-1])

## Ajuste de conversa (SFT) + avaliação

In [ ]:
# 7) SFT com o dataset HEILO (só exemplos aprovados; validação separada)
from heilo.training.pipeline import TrainingPipeline
pipe = TrainingPipeline()
split = seed_v1.dividir_sft(pipe)
print(split["manifest"])
modelo = res_pre["modelo"]
res_sft = pretrain.ajustar_conversa(modelo, split["treino"], split["val"], passos=PASSOS_SFT, lote=LOTE)
import json
arq = pretrain.exportar(modelo, Path(PASTA) / f"heilo_seed_{NOME_VERSAO}.pt",
                        {"versao": NOME_VERSAO, "passos_pretreino": res_pre["passo"],
                         "hist_pretreino": res_pre["hist"], "sft": res_sft,
                         "corpus": json.loads((corpus_dir / "meta.json").read_text())})
print("exportado:", arq, round(arq.stat().st_size / 1e6, 1), "MB")

In [ ]:
# 8) Avaliação na MESMA régua das versões anteriores + decisão automática
import json
from heilo.training.ciclos import CiclosSeed
ciclos = CiclosSeed(pipe)
dec = seed_v1.registrar(ciclos, arq, NOME_VERSAO, split["manifest"],
                        {"pretreino": {"passos": res_pre["passo"], "minutos": res_pre["minutos"],
                                       "ultimo": res_pre["hist"][-1] if res_pre["hist"] else None},
                         "sft": {k: res_sft[k] for k in ("perda_val_inicial", "melhor_perda_val")}})
print("PROMOVIDA" if dec["promovida"] else "NÃO promovida", "—", dec["motivo"])
print(ciclos.escrever_relatorio().read_text())
from heilo.models.seed.gpt import MiniGPTChat
chat = MiniGPTChat(arquivo=ciclos.vdir / NOME_VERSAO / "heilo_seed.pt")
for q in ["oi, tudo bem?", "quem é você?", "qual é o oposto de quente?", "o que é uma função em python?",
          "Responda só com o número: quanto é 2 mais 2?", "me fala sobre o Brasil"]:
    print("Você:", q, "\nHEILO:", chat.responder([{"role": "user", "content": q}], temperatura=0.7), "\n")

In [ ]:
# 9) Salvar no GitHub (versão nova + registro; a ativa só muda se foi promovida)
!cd /content/heilo && git config user.email "heilo@colab" && git config user.name "HEILO (Colab)"
!cd /content/heilo && git add -f models/seed/versions/{NOME_VERSAO} data/training/versions.json data/training/eval && git add models/seed/weights && git commit -m "HEILO Seed {NOME_VERSAO}: pré-treino PT + SFT (Colab)" || echo "nada novo"
!cd /content/heilo && git pull --rebase && git push
print("Pronto! No PC: dois cliques em atualizar_heilo.bat")